In [21]:
import os
from dotenv import load_dotenv

In [22]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated,Literal
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
import operator

In [23]:
load_dotenv()
model = ChatGoogleGenerativeAI(model = "gemini-3.6-flash")

In [24]:
class sentimentSchema(BaseModel):
    sentiment:Literal["positive","Negative"] = Field(description='sentiment of the review')

In [25]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

In [26]:
structured_model = model.with_structured_output(sentimentSchema)
structured_model2 = model.with_structured_output(DiagnosisSchema)


In [27]:
prompt = "this product is really bad"
structured_model.invoke(prompt)


sentimentSchema(sentiment='Negative')

In [28]:
class ReviewState(TypedDict):
    review : str
    sentiment : str
    diagnosis : dict
    response : str
    

In [29]:
def find_sentiment(state:ReviewState):
    prompt = f'Evaluate the sentiment of the following review - \n {state["review"]}'

    sentiment = structured_model.invoke(prompt)
    return {'sentiment':sentiment}

In [30]:
def positive_response(state:ReviewState):
    prompt = f"""write a warm thank-you message in response to the review \n -{state['review']}. Also kindly ask the user to leave the feedback on the website"""

    response = model.invoke(response).content

    return {'response':response}

In [31]:
def run_diagnosis(state:ReviewState):
    prompt = f"""Diagnose this negative review:\n\n{state['review']}\n"
    "Return issue_type, tone, and urgency.
"""
    response = structured_model2.invoke(prompt)
    return {'diagnosis':response.model_dump()}

In [32]:
def negative_response(state:ReviewState):
    diagnosis = state['diagnosis']
    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message.
"""
    response = model.invoke(prompt).content
    return { 'response':response}

In [33]:
def check_sentiment(state:ReviewState) -> Literal["positive_response","run_diagnosis"]:
    sentiment = state["sentiment"]
    if sentiment =="positive":
        return "positive_response"
    else:
        return "run_diagnosis"


In [36]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment',find_sentiment)
graph.add_node('positive_response',positive_response)
graph.add_node('negative_response',negative_response)
graph.add_node('run_diagnosis',run_diagnosis)

graph.add_edge(START,'find_sentiment')
graph.add_conditional_edges('find_sentiment',check_sentiment)

graph.add_edge('positive_response',END)
graph.add_edge('run_diagnosis','negative_response')
graph.add_edge('negative_response',END)

workflow = graph.compile()


In [37]:
initial_state = {
    'review': "I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality."
}
workflow.invoke(initial_state)

{'review': 'I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality.',
 'sentiment': sentimentSchema(sentiment='Negative'),
 'diagnosis': {'issue_type': 'Bug', 'tone': 'frustrated', 'urgency': 'high'},
 'response': [{'type': 'text',
   'text': "Subject: Resolved: Issue with [Feature/App Name]\n\nHi [User Name],\n\nThank you for bringing this issue to our attention. I want to sincerely apologize for the frustration this bug caused you today. I know how important this is to your workflow, especially given the urgency, and I completely understand how disruptive it can be when things don't work as expected.\n\nI’m glad to let you know that our engineering team has identified the root cause and deployed a fix. **The issue is now fully resolved.**\n\nTo ensure the fix takes effect on your end, please try the following:\n